# Australian dwellings per person, and per household, 1955 to 2026

A quarterly count of Australian dwellings divided by population. The ABS publishes the dwelling
count only from 2011; before that it is reconstructed here by walking the published stock backwards
with ABS dwelling completions.

Dwellings per person is not the right test of a housing shortage, because the unit that occupies a
dwelling is a household. Household counts only exist at each Census, so dwellings per household is
computed at the four censuses the ABS API carries, 2006 to 2021.

Run top to bottom with `uv run jupyter lab`. Writes `dwellings-per-person.csv` and the charts in
`img/`. The README carries the same charts and the assumptions in plain language.

In [ ]:
import io
import json
import os
import urllib.request

import matplotlib.pyplot as plt
import openpyxl
import pandas as pd

ABS = "https://data.api.abs.gov.au/rest/data/"
UA = {"User-Agent": "Mozilla/5.0"}


def sdmx(path, accept="application/vnd.sdmx.data+json"):
    req = urllib.request.Request(ABS + path, headers={**UA, "Accept": accept})
    with urllib.request.urlopen(req, timeout=90) as r:
        return json.loads(r.read())

## Sources

All free and keyless.

| Series | Source | Coverage |
| --- | --- | --- |
| Number of residential dwellings | ABS `RES_DWELL_ST`, measure 4, Australia | 2011-Q3 onward |
| Dwelling units completed | ABS `BUILDING_ACTIVITY`, measure M7, Australia | 1955-Q1 onward |
| Population at June | ABS Historical population, 2021, HPDC1 Table 2 | 1901 to 2021 |
| Population at June | ABS `ERP_Q` | 1981 to 2026 |
| Households | ABS Census, `ABS_C16_T23_TS_SA` | 2006, 2011, 2016 |
| Households | ABS Census, `C21_G35_SA2` | 2021 |

In [ ]:
# Published dwelling count.
d = sdmx("ABS,RES_DWELL_ST,1.0.0/all?format=jsondata")
st = d["data"]["structures"][0]
sdims = st["dimensions"]["series"]
ovals = st["dimensions"]["observation"][0]["values"]

count = {}
for key, ser in d["data"]["dataSets"][0]["series"].items():
    idx = [int(i) for i in key.split(":")]
    lab = {sdims[j]["id"]: sdims[j]["values"][idx[j]]["id"] for j in range(len(sdims))}
    if lab.get("MEASURE") == "4" and lab.get("REGION") == "AUS":
        for o, v in ser["observations"].items():
            count[ovals[int(o)]["id"]] = v[0] * 1000        # thousands to dwellings

print(f"{len(count)} quarters, {min(count)}, {max(count)}")

In [ ]:
# Dwelling units completed.
d = sdmx("ABS,BUILDING_ACTIVITY,1.0.0/M7.AUS..1.9.100.10.Q?format=jsondata")
ovals = d["data"]["structures"][0]["dimensions"]["observation"][0]["values"]
ser = list(d["data"]["dataSets"][0]["series"].values())[0]
completed = {ovals[int(o)]["id"]: v[0] for o, v in ser["observations"].items()}

print(f"{len(completed)} quarters, {min(completed)}, {max(completed)}")

In [ ]:
# Population at June: long historical series to 2021, ERP after.
url = "https://www.abs.gov.au/statistics/people/population/historical-population/2021/HPDC1.xlsx"
req = urllib.request.Request(url, headers=UA)
wb = openpyxl.load_workbook(
    io.BytesIO(urllib.request.urlopen(req, timeout=180).read()), read_only=True, data_only=True
)
rows = list(wb["Table 2"].iter_rows(values_only=True))
header = rows[4]
cols = [(i, h) for i, h in enumerate(header) if isinstance(h, int) and 1900 < h < 2100]
total = [r for r in rows[5:]
         if r and r[0] == "Person" and isinstance(r[1], str) and r[1].startswith("Australia")][0]
annual = {year: float(str(total[i]).replace(",", "")) for i, year in cols}

d = sdmx("ABS,ERP_Q,1.0.0/1.3.TOT.AUS.Q?format=jsondata")
ovals = d["data"]["structures"][0]["dimensions"]["observation"][0]["values"]
ser = list(d["data"]["dataSets"][0]["series"].values())[0]
erp = {}
for o, v in ser["observations"].items():
    year, q = ovals[int(o)]["id"].split("-Q")
    if q == "2":
        erp[int(year)] = v[0]

population = {**annual, **{y: p for y, p in erp.items() if y >= 2022}}
print(f"historical {min(annual)} to {max(annual)}, ERP to {max(erp)}")

## Method

1. Take the published ABS dwelling count at 2011-Q3.
2. Walk it backwards one quarter at a time: `stock(t-1) = stock(t) - completions(t) + demolitions(t)`.
3. Completions are published. Demolitions are not, so they are modelled as a constant share of the
   stock, calibrated two ways.
4. Divide by population at June for dwellings per person.
5. Divide by Census households for dwellings per household.

`dwellings` is calibrated so the stock in 1955-Q1 lands on the roughly 2.6 million the 1954 Census
counted. `dwellings_low` applies the demolition rate implied by 2011-2026, which is the floor.

In [ ]:
def qkey(s):
    year, q = s.split("-Q")
    return int(year), int(q)


quarters = sorted(set(completed) | set(count), key=qkey)

implied = [
    (completed[quarters[i]] - (count[quarters[i]] - count[quarters[i - 1]])) / count[quarters[i]]
    for i in range(1, len(quarters))
    if quarters[i] in count and quarters[i - 1] in count
]
recent_rate = sum(implied) / len(implied)
print(f"implied demolitions {recent_rate * 100:.4f}% of stock per quarter ({recent_rate * 400:.3f}%/yr)")

In [ ]:
def walk_back(rate):
    stock = {q: count[q] for q in count}
    for i in range(len(quarters) - 1, 0, -1):
        if quarters[i - 1] in count:
            continue
        stock[quarters[i - 1]] = (
            stock[quarters[i]] - completed.get(quarters[i], 0) + rate * stock[quarters[i]]
        )
    return stock


lo, hi = 0.0, 0.01
for _ in range(80):
    mid = (lo + hi) / 2
    if walk_back(mid)["1955-Q1"] < 2_600_000:
        lo = mid
    else:
        hi = mid
anchored_rate = (lo + hi) / 2

dwellings = walk_back(anchored_rate)
dwellings_low = walk_back(recent_rate)
print(f"anchored {anchored_rate * 100:.4f}%/qtr, recent {recent_rate * 100:.4f}%/qtr")
print(f"1955-Q1: {dwellings['1955-Q1']:,.0f} anchored, {dwellings_low['1955-Q1']:,.0f} low")

In [ ]:
rows = []
for q in quarters:
    year, quarter = q.split("-Q")
    year = int(year)
    pop = population.get(year)
    june = quarter == "2" and pop
    rows.append({
        "quarter": q,
        "source": "ABS" if q in count else "backcast",
        "completions": int(completed.get(q, 0)),
        "dwellings": int(dwellings[q]),
        "dwellings_low": int(dwellings_low[q]),
        "population_june": int(pop) if june else None,
        "dwellings_per_person": round(dwellings[q] / pop, 4) if june else None,
        "dwellings_per_person_low": round(dwellings_low[q] / pop, 4) if june else None,
    })

df = pd.DataFrame(rows)
df.to_csv("dwellings-per-person.csv", index=False)

june = df[df.population_june.notna()].copy()
june["year"] = june.quarter.str[:4].astype(int)
june[june.year % 5 == 0][["quarter", "source", "dwellings", "population_june", "dwellings_per_person"]]

## Households

Households come from the Census, so they exist at five-yearly points and the ABS API publishes them
by region. Australia is selected with `REGION_TYPE=AUS`, and the household-size cross-tab is summed
to give both the household count and the average size.

In [ ]:
def census_households(flow, version, size_dim, hh_dim):
    d = sdmx(f"ABS,{flow},{version}/...AUS.?format=jsondata")
    st = d["data"]["structures"][0]
    sdims = st["dimensions"]["series"]
    ovals = st["dimensions"]["observation"][0]["values"]
    out = {}
    for key, ser in d["data"]["dataSets"][0]["series"].items():
        idx = [int(i) for i in key.split(":")]
        lab = {sdims[i]["id"]: sdims[i]["values"][idx[i]]["id"] for i in range(len(sdims))}
        size = lab.get(size_dim)
        # The 2016 codelists use TOT for total, the 2021 ones use _T.
        if lab.get(hh_dim) not in ("TOT", "_T") or size in (None, "NA"):
            continue
        for o, v in ser["observations"].items():
            year = int(ovals[int(o)]["id"])
            households, persons = out.get(year, (0, 0))
            if size in ("TOT", "_T"):
                households = v[0]
            elif size.isdigit():
                persons += int(size) * v[0]
            out[year] = (households, persons)
    return out


households = census_households("ABS_C16_T23_TS_SA", "1.0.0", "NPRD_2016", "HHCD_2016")
households.update(census_households("C21_G35_SA2", "1.0.0", "NPRD", "HHCD"))
print({y: f"{h:,}" for y, (h, _) in sorted(households.items())})

In [ ]:
per_household = june[june.year.isin(households)].copy()
per_household["households"] = per_household.year.map({y: h for y, (h, _) in households.items()})
per_household["average_size"] = per_household.year.map(
    {y: round(persons / h, 3) for y, (h, persons) in households.items()}
)
per_household["dwellings_per_household"] = (
    per_household.dwellings / per_household.households
).round(3)
per_household[["year", "households", "average_size", "dwellings", "dwellings_per_household"]]

In [ ]:
os.makedirs("img", exist_ok=True)
grey = june.year.max()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(june.year, june.dwellings_per_person, label="anchored to the 1954 Census (2.6m in 1955)")
ax.plot(june.year, june.dwellings_per_person_low, linestyle="--",
        label="recent demolition rate (floor)")
ax.axvspan(2011.5, grey, color="grey", alpha=0.08, label="ABS published count")
ax.set_xlabel("June year")
ax.set_ylabel("dwellings per person")
ax.set_title("Australian dwellings per person, 1955 to 2026")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("img/dwellings-per-person.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(june.year, june.dwellings / 1e6, label="anchored to the 1954 Census")
ax.plot(june.year, june.dwellings_low / 1e6, linestyle="--", label="recent demolition rate")
ax.axvspan(2011.5, grey, color="grey", alpha=0.08, label="ABS published count")
ax.annotate("1954 Census anchor, 2.6m", xy=(1955, 2.6), xytext=(1963, 1.55),
            arrowprops={"arrowstyle": "->"})
ax.set_xlabel("June year")
ax.set_ylabel("dwellings (millions)")
ax.set_title("Dwelling count: how the demolition assumption fans out backwards")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("img/demolition-assumption.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(per_household.year, per_household.dwellings_per_household, marker="o", linewidth=2)
for _, r in per_household.iterrows():
    ax.annotate(f"{r.dwellings_per_household:.3f}", (r.year, r.dwellings_per_household),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax.set_xticks(list(per_household.year))
ax.set_ylim(1.0, 1.25)
ax.set_xlabel("Census year")
ax.set_ylabel("dwellings per household")
ax.set_title("Dwellings per household, 2006 to 2021\nflat at about 1.16, so the stock kept pace with household formation")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("img/dwellings-per-household.png", dpi=150)
plt.show()

## Caveats

- Everything before 2011-Q3 is reconstructed. From 2011-Q3 onward it is the published ABS count.
- Demolitions are the only free parameter, and a constant rate is assumed. If the true rate varied,
  the levels before 2011 are off, most at the oldest end.
- The 2.6 million anchor for 1955 comes from the 1954 Census and should be verified.
- Households exist only at Censuses, so dwellings per household is four points, not a series, and
  the API carries households from 2006 only. The pre-2006 path, which is where a squeeze would
  hide, needs the older Census volumes, which are PDFs rather than data.
- The dwellings numerator is the total stock including unoccupied dwellings; households are occupied
  dwellings. So dwellings per household is above one by construction, and roughly 1.16 is the slack
  the market holds.
- Rejected alternative: dividing the ABS real (chain-volume) capital stock of dwellings by real
  capital per dwelling. That ratio is stable over 2012-2025 (0.9% spread) but cannot be checked
  earlier, so it is the same undated assumption in dollars rather than dwelling units.